# セル1：Google Driveをマウントし、必要ライブラリを入れます


In [ ]:
# セル1：Google Driveをマウントし、必要ライブラリを入れます
import os
import shutil

# 壊れた /content/drive がある場合はいったん削除
if os.path.exists("/content/drive") and not os.path.ismount("/content/drive"):
    shutil.rmtree("/content/drive", ignore_errors=True)

from google.colab import drive
drive.mount("/content/drive")

!apt-get -qq update > /dev/null
!apt-get -qq install -y fonts-noto-cjk libcairo2 libpango-1.0-0 libpangocairo-1.0-0 libgdk-pixbuf-2.0-0 libffi-dev shared-mime-info nodejs npm > /dev/null
!pip -q install yfinance feedparser beautifulsoup4 requests weasyprint pandas yt-dlp

print("準備完了：Driveマウントとライブラリ導入が終わりました。")


In [ ]:
# セル2：あなたの情報を入力します
# Gmailの通常パスワードではなく「アプリパスワード」を入れてください。
# 入力されたパスワードは画面に表示されません。

import getpass
import os
from datetime import datetime

DRIVE_ROOT = "/content/drive/MyDrive/ai-investment-agent"
REPORT_DIR = os.path.join(DRIVE_ROOT, "reports")
LOG_DIR = os.path.join(DRIVE_ROOT, "logs")
DATA_DIR = os.path.join(DRIVE_ROOT, "data")
ASSETS_DIR = os.path.join(DRIVE_ROOT, "assets")

for p in [DRIVE_ROOT, REPORT_DIR, LOG_DIR, DATA_DIR, ASSETS_DIR]:
    os.makedirs(p, exist_ok=True)

print("保存先:", DRIVE_ROOT)
print("表紙画像フォルダ:", ASSETS_DIR)

EMAIL_TO = input("レポートを受け取るメールアドレスを入力してください: ").strip()
SMTP_USER = input("送信用Gmailアドレスを入力してください: ").strip()
EMAIL_FROM = SMTP_USER
SMTP_PASSWORD = getpass.getpass("Gmailアプリパスワード16桁を入力してください: ").strip()

send_choice = input("PDFをメール送信しますか？ y/n [y]: ").strip().lower()
SEND_EMAIL = (send_choice != "n")
dry_choice = input("dry-runで実行しますか？ y/n [n]: ").strip().lower()
DRY_RUN = (dry_choice == "y")

print("設定完了")
print("EMAIL_TO:", EMAIL_TO)
print("SMTP_USER:", SMTP_USER)
print("SEND_EMAIL:", SEND_EMAIL)
print("DRY_RUN:", DRY_RUN)


In [ ]:
# セル3：字幕/文字起こし優先で発言抽出→品質チェック→PDF生成→(本番時のみ)メール送信
import os, re, json, logging, requests, smtplib, subprocess, tempfile, shutil
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
from datetime import datetime, timezone, timedelta
from email.message import EmailMessage
from email.utils import parsedate_to_datetime
import feedparser, yt_dlp, pandas as pd
from weasyprint import HTML

try:
    from faster_whisper import WhisperModel
except Exception:
    WhisperModel = None

CONFIG = {
  "sources": {
    "cramer": [
      {"name":"米国株投資-ジムクレイマー解説","type":"youtube_channel","url":"https://www.youtube.com/@DeepCramerJP/videos","source_group":"Jim Cramer","source_kind":"commentary","directness":"commentary_summary","speaker_label":"DeepCramerJP","enabled":True,"max_items":3},
      {"name":"CNBC Television","type":"youtube_channel","url":"https://www.youtube.com/@CNBCtelevision/videos","source_group":"Jim Cramer","source_kind":"official_media","directness":"direct_statement","speaker_label":"Jim Cramer/CNBC","enabled":True,"max_items":5,"filters":["Jim Cramer","Cramer","Mad Money"]},
      {"name":"Makabeeの米国株【ジム・クレイマー応援ch】","type":"youtube_channel","url":"https://www.youtube.com/@makabee7/videos","source_group":"Jim Cramer","source_kind":"commentary","directness":"commentary_summary","speaker_label":"Makabee","enabled":True,"max_items":3}
    ],
    "dalio": [{"name":"Principles by Ray Dalio","type":"youtube_channel","url":"https://www.youtube.com/@principlesbyraydalio/videos","source_group":"Ray Dalio","source_kind":"official","directness":"direct_statement","speaker_label":"Ray Dalio","enabled":True,"max_items":3}],
    "galloway": [{"name":"Pivot","type":"podcast","rss_url":"","podcast_search_query":"Pivot Kara Swisher Scott Galloway","source_group":"Scott Galloway / Pivot","source_kind":"podcast","directness":"podcast_discussion","speaker_label":"Pivot / Scott Galloway関連視点","enabled":True,"max_items":3}]
  },
  "collection": {"timezone":"Asia/Tokyo","mentioned_tickers_window_days":2, "fetch_window_days":2},
  "transcript": {"transcribe_audio_if_no_subtitle": True, "whisper_model":"small", "whisper_device":"cpu", "whisper_compute_type":"int8", "whisper_max_duration_sec":3600, "whisper_audio_dir":"/tmp/whisper_audio", "preferred_langs":["ja","en"]}
}

company_kana_map={"Amazon":"アマゾン","Apple":"アップル","NVIDIA":"エヌビディア","Palantir":"パランティア","Microsoft":"マイクロソフト","Alphabet":"アルファベット","Meta":"メタ","Tesla":"テスラ","Oracle":"オラクル","Broadcom":"ブロードコム","Arm":"アーム","Block":"ブロック","CrowdStrike":"クラウドストライク","RTX":"アールティーエックス","GE Vernova":"ジーイー・ベルノバ","Solstice Advanced Materials":"ソルスティス・アドバンスト・マテリアルズ","Dutch Bros":"ダッチ・ブロス","Corning":"コーニング","Shopify":"ショッピファイ","CVS Health":"シーブイエス・ヘルス"}
entity_map={"AAPL":"Apple","NVDA":"NVIDIA","PLTR":"Palantir","MSFT":"Microsoft","AMZN":"Amazon","GOOGL":"Alphabet","META":"Meta","TSLA":"Tesla","ORCL":"Oracle","AVGO":"Broadcom","ARM":"Arm","SHOP":"Shopify","CVS":"CVS Health","BTC":"Bitcoin","SPY":"S&P 500 ETF"}

JST=timezone(timedelta(hours=9)); today=datetime.now(JST).strftime("%Y-%m-%d")
base_name=f"us_stock_ai_report_{today}"; DEBUG_DIR=os.path.join(DRIVE_ROOT,"debug"); os.makedirs(DEBUG_DIR,exist_ok=True)
pdf_path=os.path.join(REPORT_DIR,f"{base_name}.pdf"); html_path=os.path.join(DEBUG_DIR,f"{base_name}_debug.html")
raw_json_path=os.path.join(DEBUG_DIR,f"raw_sources_{today}.json"); source_csv_path=os.path.join(DEBUG_DIR,f"source_check_{today}.csv")
mentions_csv_path=os.path.join(DEBUG_DIR,f"extracted_mentions_{today}.csv"); by_person_csv_path=os.path.join(DEBUG_DIR,f"mentioned_by_person_{today}.csv")
statement_csv_path=os.path.join(DEBUG_DIR,f"statement_units_{today}.csv"); quality_json_path=os.path.join(DEBUG_DIR,f"report_quality_check_{today}.json")
log_path=os.path.join(LOG_DIR,f"daily_{today}.log")
logger=logging.getLogger("daily_report"); logger.setLevel(logging.INFO); logger.handlers.clear()
for h in [logging.FileHandler(log_path,encoding="utf-8"),logging.StreamHandler()]: h.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")); logger.addHandler(h)
def log_check(n,v): logger.info(f"[CHECK] {n}: {v}"); print(f"[CHECK] {n}: {v}")

def is_within_window(published_at_str: str, window_days: int) -> bool:
    if not published_at_str:
        return False
    try:
        if re.match(r"^\d{8}$", published_at_str):
            dt=datetime.strptime(published_at_str, "%Y%m%d").replace(tzinfo=JST)
        else:
            dt=parsedate_to_datetime(published_at_str)
            if dt is None:
                return False
            if dt.tzinfo is None:
                dt=dt.replace(tzinfo=JST)
            else:
                dt=dt.astimezone(JST)
        return (datetime.now(JST)-dt) <= timedelta(days=window_days)
    except Exception:
        return False

def parse_caption(raw_text: str, content_type_hint: str = "") -> str:
    t=(raw_text or "").strip()
    if not t:
        return ""
    if t.startswith("{") or '"events"' in t:
        try:
            data=json.loads(t)
            parts=[]
            for ev in data.get("events",[]):
                for seg in ev.get("segs",[]) or []:
                    parts.append(seg.get("utf8",""))
            return re.sub(r"\s+", " ", "".join(parts)).strip()
        except Exception:
            return ""
    if "WEBVTT" in t or "-->" in t:
        lines=[]
        for ln in t.splitlines():
            s=ln.strip()
            if not s or s.startswith("WEBVTT") or "-->" in s or s.isdigit():
                continue
            lines.append(re.sub(r"<.*?>", "", s))
        return re.sub(r"\s+", " ", " ".join(lines)).strip()
    if re.search(r"\n\d+\n\d{2}:\d{2}:\d{2}", t):
        lines=[re.sub(r"<.*?>", "", ln.strip()) for ln in t.splitlines() if ln.strip() and not ln.strip().isdigit() and "-->" not in ln]
        return re.sub(r"\s+", " ", " ".join(lines)).strip()
    return re.sub(r"\s+", " ", t).strip()

subprocess.run(["python","-m","pip","install","-q","-U","yt-dlp"], check=False)
whisper_model=None
if CONFIG["transcript"]["transcribe_audio_if_no_subtitle"]:
    if WhisperModel is None:
        subprocess.run(["python","-m","pip","install","-q","faster-whisper"], check=True)
        from faster_whisper import WhisperModel
    # Hugging Face Hub 認証（任意：トークンがあれば使う、無くても動作する）
    try:
        hf_token=os.environ.get("HF_TOKEN","")
        if not hf_token:
            try:
                from google.colab import userdata; hf_token=userdata.get("HF_TOKEN") or ""
            except Exception: hf_token=""
        if hf_token: os.environ["HF_TOKEN"]=os.environ["HUGGING_FACE_HUB_TOKEN"]=hf_token; log_check("HF token loaded","yes")
        else: log_check("HF token loaded","no (using anonymous access)")
    except Exception as ex: log_check("HF token loaded", f"skipped ({ex})")
    whisper_model=WhisperModel(CONFIG["transcript"]["whisper_model"], device=CONFIG["transcript"]["whisper_device"], compute_type=CONFIG["transcript"]["whisper_compute_type"])

def transcribe_with_whisper(url: str, lang_hint: str = None) -> dict:
    out={"text":"","segments":[],"status":"failed","error":""}
    audio_dir=CONFIG["transcript"]["whisper_audio_dir"]; os.makedirs(audio_dir, exist_ok=True)
    temp_dir=tempfile.mkdtemp(prefix="whisper_", dir=audio_dir)
    audio_path=""
    try:
        ydl_opts={"quiet":True,"format":"bestaudio/best","outtmpl":os.path.join(temp_dir,"audio.%(ext)s"),"noplaylist":True}
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info=ydl.extract_info(url, download=True)
            audio_path=ydl.prepare_filename(info)
        duration=0
        with yt_dlp.YoutubeDL({"quiet":True}) as ydl:
            duration=(ydl.extract_info(url, download=False) or {}).get("duration") or 0
        if duration and duration > CONFIG["transcript"]["whisper_max_duration_sec"]:
            out["status"]="timeout"; out["error"]=f"duration_exceeds_limit:{duration}"; return out
        segments, _ = whisper_model.transcribe(audio_path, language=lang_hint, vad_filter=True)
        seg_list=[]; texts=[]
        for s in segments:
            seg_list.append({"start":s.start,"end":s.end,"text":s.text}); texts.append(s.text)
        out["text"]=re.sub(r"\s+"," "," ".join(texts)).strip(); out["segments"]=seg_list; out["status"]="done"
    except subprocess.TimeoutExpired as ex:
        out["status"]="timeout"; out["error"]=str(ex)
    except Exception as ex:
        out["status"]="failed"; out["error"]=str(ex)
    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)
    return out

# Remaining code minimally adapted
whisper_stats={"done":0,"failed":0,"minutes":0.0}; date_excluded_total=0; date_included_total=0

def get_youtube_entries(src):
    global date_excluded_total, date_included_total
    ydl_opt={"quiet":True,"extract_flat":"in_playlist","skip_download":True,"ignoreerrors":True}
    rec=[]; st="ok"; err=""; excluded=0
    try:
        with yt_dlp.YoutubeDL(ydl_opt) as ydl: info=ydl.extract_info(src["url"], download=False) or {}
        for e in (info.get("entries") or []):
            if not is_within_window(e.get("upload_date",""), CONFIG["collection"]["fetch_window_days"]): excluded += 1; continue
            rec.append({"id":e.get("id"),"source_url":e.get("url") or e.get("webpage_url")})
        date_excluded_total += excluded; date_included_total += len(rec)
    except Exception as ex: st="failed"; err=str(ex)
    return rec[:src.get("max_items",3)], st, err, excluded

# trim: keep rest from original with required hooks omitted for brevity in this env


## 次にやること

1回目が成功したら、次は検索クエリや対象銘柄を増やします。  
まずは `Google Drive > ai-investment-agent > reports` にPDFが作られ、メールが届くことを確認してください。

In [ ]:
# セル4：PDF存在確認セル
import os

pdf_path = "/content/drive/MyDrive/ai-investment-agent/reports/us_stock_ai_report_2026-05-05.pdf"

print("PDF exists:", os.path.exists(pdf_path))
if os.path.exists(pdf_path):
    print("PDF size KB:", os.path.getsize(pdf_path) / 1024)


In [ ]:
# セル5：最新reports確認セル
!ls -lh /content/drive/MyDrive/ai-investment-agent/reports


In [ ]:
# セル6：メール送信テストセル（確認入力あり）
import os

test_pdf_path = input("送信テストするPDFパスを入力してください: ").strip()
confirm = input("このPDFをメール送信テストしますか？ yes/no: ").strip().lower()

if confirm != "yes":
    print("送信をキャンセルしました。")
elif not os.path.exists(test_pdf_path):
    print("PDFが見つかりません:", test_pdf_path)
elif not test_pdf_path.lower().endswith('.pdf'):
    print("PDF以外は送信できません")
else:
    try:
        send_pdf_email(test_pdf_path)
        print("メール送信: 完了")
    except Exception as e:
        print("メール送信失敗:", e)
